# Tests des fonctions de calcul de la saturation

In [18]:
import json
from datetime import date, timedelta

import pandas as pd
from e2_e3_e6 import (
    e2,
    e3,
    e6,
    # filter_statuses_sessions,
    #get_chunked_state_grp,
    #get_chunked_state_poc,
    #get_chunked_state_pools,
    # get_sampled_state_poc,
    # get_state_poc_for_chunk,
)
from pandas import NamedAgg

#from saturation_image_quali_prod import (
from utils import (
    filter_sessions_duration,
    # to_sampled_state_grp,
    # to_state_grp,
    # to_state_poc,
)

SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
#day = date(2026,7, 14)
#day = date(2026,8, 1)
day = date(2026,8, 18)
#day = date(2026,8, 25)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

In [6]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for stations and pools."""
    e1_statics = pd.read_csv("../source/tests_DMR/aires_pdc_2026-07-25.csv")[[ID_POOL, ID_STATION]].drop_duplicates()
    return e1_statics

In [7]:
e1_statics = read_statics_pools(day, MIN_POWER)
e1_statics

,id_pool,id_station_itinerance
0,A000001,FRHPCPNF080371TIERSTOTEM
13,A000002,FRTSLP5670
14,A000002,FRIOYP13531046
29,A000003,FRIOYP13530804
51,A000004,FRFASP11568703
...,...,...
7112,A001871,NaN
7113,A001872,NaN
7114,A001873,NaN
7115,A001874,NaN


## test qualicharge

In [19]:
from datetime import datetime
    
samples_per_day = SAMPLES
min_power = MIN_POWER
chunk_size = 200

min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)

statuses = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
sessions = filter_sessions_duration(
    sessions_s3, min_duration=min_duration, max_duration=max_duration
)
#statics = read_statics(day, MIN_POWER)
#statics = statics[statics["puissance_nominale"] >= min_power]

sessions_poc = (
    sessions.groupby(ID_POC)
    .agg(
        sessions_nb=NamedAgg("energy", "count"),
        energy_cum=NamedAgg("energy", "sum"),
    )
    .reset_index()
)
statuses.memory_usage(deep=True).sum() / 1024**2, sessions_s3.memory_usage(deep=True).sum() / 1024**2


(np.float64(154.7748966217041), np.float64(30.652545928955078))

In [20]:
len(statuses), len(sessions_s3)

(755913, 152729)

In [5]:
# e2 indicator
sampled_state_poc, state_poc = get_chunked_state_poc(
    statics, day, samples_per_day, chunk_size, sessions, statuses
)
indicators_e2 = e2(
    #environment,
    state_poc,
    sessions_poc,
    day,
    #create_artifact,
    #persist,
)

In [10]:
state_poc.sort_values(by=[ID_POC])

,id_pdc_itinerance,occupe,occupe_max,hors_service,libre,pseudo_libre,pseudo_occupe
5117,FR3R3E10001456611,70.0,60.0,0.0,1370.0,70.0,0.0
12681,FR3R3E10001456612,80.0,45.0,0.0,1360.0,80.0,0.0
14498,FRALDE100541,145.0,55.0,0.0,1295.0,0.0,0.0
14011,FRALDE100551,170.0,60.0,0.0,1270.0,0.0,15.0
12350,FRALDE100552,90.0,45.0,0.0,1350.0,0.0,0.0
...,...,...,...,...,...,...,...
8826,FRZUNEFR8601ER06,220.0,55.0,15.0,1205.0,0.0,5.0
2248,FRZUNEFR8801ER01,175.0,60.0,0.0,1265.0,0.0,0.0
14337,FRZUNEFR8801ER02,140.0,55.0,0.0,1300.0,0.0,0.0
5902,FRZUNEFR8801ER03,65.0,35.0,0.0,1375.0,0.0,5.0


In [11]:
# e3 indicator
state_station = get_chunked_state_grp(
    statics,
    sampled_state_poc,
    chunk_size,
    ID_STATION,
    SAMPLES,
    SATURATION_RATIO,
    OVERLOAD_RATIO,
    add_full_use=True,
    add_latency=True,
)
sessions_stations = pd.merge(
    statics[[ID_POC, ID_STATION]], sessions_poc, on=ID_POC, how="left"
).fillna(0)
info_sessions_stations = (
    sessions_stations[[ID_STATION, "sessions_nb", "energy_cum"]]
    .groupby(ID_STATION)
    .sum()
    .reset_index()
)
indicators_e3 = e3(
    #environment,
    state_station,
    info_sessions_stations,
    day,
    #create_artifact,
    #persist,
)

In [12]:
state_station.sort_values(by=[ID_STATION])

,id_station_itinerance,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
2869,FR3R3P89882136,2,0.0,1290.0,0.0,0.0,0.0,150.0,0.0,0.0,0.0
1992,FRALDPFR00916,8,0.0,1220.0,0.0,0.0,0.0,220.0,0.0,0.0,0.0
3219,FRALDPFR00950,8,0.0,995.0,0.0,0.0,5.0,440.0,0.0,0.0,0.0
917,FRALLPGO000007,6,0.0,830.0,340.0,205.0,170.0,235.0,60.0,60.0,230.0
3922,FRALLPGO000013,10,0.0,395.0,0.0,0.0,40.0,1005.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
3043,FRZUNP6023950095875504781,8,0.0,570.0,120.0,25.0,60.0,785.0,15.0,55.0,55.0
1820,FRZUNP6927750076048540479,8,0.0,1090.0,0.0,0.0,0.0,350.0,0.0,0.0,0.0
366,FRZUNP7329346578064027187,25,0.0,175.0,0.0,0.0,0.0,1265.0,0.0,0.0,0.0
367,FRZUNP8610050047391683219,6,0.0,1000.0,235.0,20.0,115.0,305.0,20.0,60.0,150.0


In [13]:
# e6 indicator
#pools_statics = get_station_pool_for_day(day, environment)
pools_statics = read_statics_pools(day, MIN_POWER)
pools_stations = pools_statics[[ID_POOL, ID_STATION]].drop_duplicates()
pools_pocs = pools_statics.merge(statics, on=ID_STATION, how="left")[[ID_POOL, ID_POC]]

state_pool = get_chunked_state_pools(
    statics,
    pools_stations,
    state_station,
    sampled_state_poc,
    chunk_size,
    SAMPLES,
    SATURATION_RATIO,
    OVERLOAD_RATIO,
    add_full_use=True,
    add_latency=True,
)
sessions_pools = pd.merge(
    pools_pocs, sessions_poc, on=ID_POC, how="left"
).fillna(0)
info_sessions_pools = (
    sessions_pools[[ID_POOL, "sessions_nb", "energy_cum"]]
    .groupby(ID_POOL)
    .sum()
    .reset_index()
)
indicators_e6 = e6(
    #environment,
    state_pool,
    info_sessions_pools,
    day,
    #create_artifact,
    #persist,
)


In [18]:
state_pool.sort_values(by=[ID_POOL])[1690:1710]

,id_pool,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
1693,A001691,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1694,A001692,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1695,A001693,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1696,A001694,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1697,A001695,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1698,A001696,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1699,A001697,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
490,A001698,12.0,0.0,455.0,320.0,170.0,120.0,695.0,50.0,60.0,310.0
1700,A001699,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1701,A001700,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
